In [ ]:
import os
from pathlib import Path


def find_files(directory, extensions=None):
    """Recursively yield files under `directory` matching `extensions`."""
    if extensions is None:
        extensions = ['.mp4', '.mkv', '.avi', '.mov', '.flv', '.wmv', '.ts']
    directory = Path(directory)
    if not directory.exists():
        print(f'Directory not found: {directory}')
        return
    for root, dirs, files in os.walk(str(directory)):
        for fn in files:
            if any(fn.lower().endswith(ext.lower()) for ext in extensions):
                yield Path(root) / fn


def load_list_file(list_path):
    """Load newline-separated list of paths or filenames from a text file into a set."""
    p = Path(list_path)
    if not p.exists():
        print(f'List file not found: {p}')
        return set()
    with p.open('r', encoding='utf-8') as fh:
        return set(line.strip() for line in fh if line.strip())


# Try to import a MySQL client; if unavailable, DB checks will be skipped unless user provides an alternative implementation.
try:
    import pymysql
except Exception:
    pymysql = None


def check_in_mysql_db(filename, host='localhost', user=None, password=None, db=None, table='files', column='path', port=3306):
    """Check whether `filename` exists in a MySQL table's `column`. Requires `pymysql`."""
    if pymysql is None:
        raise RuntimeError('pymysql is not installed in this environment')
    conn = pymysql.connect(host=host, user=user, password=password, database=db, port=port)
    try:
        with conn.cursor() as cur:
            cur.execute(f"SELECT 1 FROM `{table}` WHERE `{column}`=%s LIMIT 1", (filename,))
            return cur.fetchone() is not None
    finally:
        conn.close()


def check_files_against_mysql(directory, mysql_config=None, list_file=None, extensions=None):
    """Scan `directory` and check each matching file against either `list_file` or a MySQL table.

    Yields dicts: {'path': str, 'listed_in': 'list_file'|'mysql_db'|None, 'detail': optional}
    """
    listed = set()
    if list_file:
        listed = load_list_file(list_file)
    use_db = mysql_config is not None
    for p in find_files(directory, extensions=extensions):
        pstr = str(p)
        listed_in = None
        # Check list file by full path or basename
        if pstr in listed or p.name in listed:
            listed_in = 'list_file'
            detail = 'matched list file (by fullpath or basename)'
        elif use_db:
            try:
                ok = check_in_mysql_db(pstr, **mysql_config)
                if ok:
                    listed_in = 'mysql_db'
                    detail = 'found in mysql table'
                else:
                    detail = 'not found in mysql table'
            except Exception as e:
                listed_in = None
                detail = f'mysql_check_error: {e}'
        else:
            detail = 'no check performed'
        yield {'path': pstr, 'listed_in': listed_in, 'detail': detail}


# Example usage:
# 1) Using a text list exported from MySQL:
# for r in check_files_against_mysql('/media/Media 1', list_file='mysql_listing.txt'):
#     print(r)

# 2) Using a live MySQL database (requires pymysql and correct credentials):
# mysql_cfg = {'host':'db.example','user':'me','password':'secret','db':'mydb','table':'files','column':'path'}
# for r in check_files_against_mysql('/media/Media 1', mysql_config=mysql_cfg):
#     print(r)
